Here I test the theory about Euler Maruyama scheme using the Ornstein–Uhlenbeck process.

In particular, I used $ dX_t = -0.5X_tdt + dW_t $

The theoretical limit is $X_\infty\sim\mathcal{N}\left(0,\text{Id}\right)$

In the following code, I implement this scheme with schedule $t_k=\frac{0.02}{k^{0.3}}$ and on $200$ samples.

Finally, I compare, in an animation, the mean and variance between current empirical state and theoretical limit. 

In [ ]:
# Test with Ornstein-Uhlenbeck process

import numpy as np
from zhai2022.sde import Model
from zhai2022.sde.euler_maruyama import EulerMaruyama
from scipy.stats import chi2

# Define drift, diffusion, and initial state functions
def drift(X, t):
    return -0.5 * X

def diffusion(X, t):
    return np.tile(np.eye(X.shape[1]), (X.shape[0], 1, 1))

def initial_state(n_samples):
    return np.ones((n_samples, 2)) * 5

# Create a Model instance
model = Model(drift, diffusion, initial_state, initial_time=0.0, n_dim=2)

dt_schedule = np.arange(1, 5001)**(-0.3) * 0.02
euler_maruyama = EulerMaruyama(model, dt_schedule)
X = euler_maruyama.get_trajectory(n_samples=1000)  # shape: (n_time_steps, n_samples, n_dim)

# mean and covariance at time t
mean_at = np.mean(X, axis=1, keepdims=True)  # shape: (n_time_steps, 1, n_dim)
covariance_at = np.einsum('tsi, tsj -> tij', X-mean_at, X-mean_at) / (X.shape[1]-1)  # shape: (n_time_steps, n_dim, n_dim)

# covariance ellipse parameters at final time
p = 0.95

# size of the ellipse
chi2_val = chi2.ppf(p, df=2)
r = np.sqrt(chi2_val)
# compute eigenvalues and eigenvectors of the covariance matrix at final time
eigenvals, eigenvecs = np.linalg.eigh(covariance_at)
lambda1, lambda2 = eigenvals[:, 0], eigenvals[:, 1]
v1, v2 = eigenvecs[:, :, 0], eigenvecs[:, :, 1]
# Semi-axes lengths
a = r * np.sqrt(lambda1)
b = r * np.sqrt(lambda2)
# Angle of the ellipse (in degrees)
angle_rad = np.arctan2(v1[:,1], v1[:,0])
angle_deg = np.degrees(angle_rad)

# make an animation
from matplotlib import pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib import animation

T = np.sum(dt_schedule)
dt_mean = T / len(dt_schedule)

# make the animation
fig, ax = plt.subplots()
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal', 'box')
ax.set_title('Ornstein-Uhlenbeck Process Simulation')

# scatter plot of samples
scat = ax.scatter(X[:, 0, 0], X[:, 0, 1], s=5, color='blue', alpha=0.5, label='Samples')
# mean point
mean_scat = ax.scatter(mean_at[:, 0, 0], mean_at[:, 0, 1], color='red', s=25, marker='x', label='Empirical result')
# insert the covariance ellipse at the final time
ell = Ellipse(
    xy=(mean_at[0,0,0], mean_at[0,0,1]),           # center
    width=2*a[0],           # width = 2 * major semi-axis
    height=2*b[0],          # height = 2 * minor semi-axis
    angle=angle_deg[0],     # angle of rotation in degrees
    edgecolor='red',
    facecolor='none',
    linewidth=2,
)
ax.add_patch(ell)

# theoretical limit
meantheo_scat = ax.scatter(0, 0, color='green', s=25, marker='x', label='Theoretical limit')
elltheo = Ellipse(
    xy=(0,0),
    width=2*r,
    height=2*r,
    angle=angle_deg[0],
    edgecolor='green',
    facecolor='none',
    linewidth=2,
)
ax.add_patch(elltheo)

ax.legend(
    loc='lower left',
    fontsize='small',
)

def update(frame):
    scat.set_offsets(X[frame, :, :])
    mean_scat.set_offsets(mean_at[frame, 0, :])
    ell.center = (mean_at[frame, 0, 0], mean_at[frame, 0, 1])
    ell.width = 2 * a[frame]
    ell.height = 2 * b[frame]
    ell.angle = angle_deg[frame]
    return scat, mean_scat, ell
ani = animation.FuncAnimation(fig, update, frames=X.shape[0], interval=int(dt_mean*1000), blit=True)
ani.save('OU_process.mp4', writer='ffmpeg', fps=int(1/dt_mean), dpi=400)
plt.close()

# 2D Ring density

In this example the SDE is:

$ dX_t = \left(-4X_t\left(\left\|X_t\right\|^2-1\right)+RX_t\right)dt + \sigma dW_t $

where $R$ is an antisymmetric matrix and $\sigma$ is a constant. In particular $\sigma\in\mathbb{R}$ (or $\sigma \parallel \text{Id}$)

The associated Fokker Planck equation is:

$ u_t = - \sum_i (f_i u)_{x_i} + \frac12\sum_{ij} (\Sigma_{ij}u)_{x_ix_j}$

where $f(x,t)=-4x\left(\left\|x\right\|^2-1\right)+Rx$ and $\Sigma(x,t)=\sigma(x,t)^T\sigma(x,t)$

We observe that $f(x,t) = -\nabla_x V\left(\left\|x\right\|^2\right)+Rx$ where $V(r^2)=\left(r^2-1\right)^2$

The steady state has form $u(x) = \frac{1}{K}e^{\alpha V(\left\|x\right\|^2)}$, indeed:

- $\sum_i (f_i u)_{x_i} = \sum_i u \partial_{x_i} f_i + u_{x_i} f_i = u \nabla_x \cdot f + f \cdot \nabla_x u$
- $\nabla_x \cdot f = \nabla_x \cdot \left(-\nabla_x V\left(\left\|x\right\|^2\right)+Rx\right) = - \Delta_x V\left(\left\|x\right\|^2\right)$
- $\nabla_x u = \alpha \nabla_x V(\left\|x\right\|^2)u $
- $f \cdot \nabla_x u = \left(-\nabla_x V\left(\left\|x\right\|^2\right)+Rx\right) \cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) = -\nabla_x V\left(\left\|x\right\|^2\right)\cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) + Rx\cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right)$
- $Rx\cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) = \alpha u Rx\cdot \nabla_x V(\left\|x\right\|^2) = \alpha u Rx\cdot\left(\nabla_{r^2}V|_{r^2=\left\|x\right\|^2}x\right) $. $R$ is an antisymmetric matrix so: $Rx\cdot x = 0$, finally $Rx\cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right)=0$
- $f \cdot \nabla_x u = -\nabla_x V\left(\left\|x\right\|^2\right)\cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) + Rx\cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) = -\nabla_x V\left(\left\|x\right\|^2\right)\cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) = -\alpha u \left\|\nabla_x V(\left\|x\right\|^2)\right\|^2$
- $\sum_i (f_i u)_{x_i} = u \nabla_x \cdot f + f \cdot \nabla_x u = u \left(- \Delta_x V\left(\left\|x\right\|^2\right)\right) - \alpha u \left\|\nabla_x V(\left\|x\right\|^2)\right\|^2 = -u\Delta_x V\left(\left\|x\right\|^2\right) - \alpha u \left\|\nabla_x V(\left\|x\right\|^2)\right\|^2$
- $\sum_{ij} (\Sigma_{ij}u)_{x_ix_j} = \sum_{ij} u\partial_{x_ix_j}\Sigma_{ij} + u_{x_i}\partial_{x_j}\Sigma_{ij} + u_{x_j}\partial_{x_i}\Sigma_{ij}+u_{x_ix_j}\Sigma_{ij}$. I observe that $\Sigma$ does not depend on $x$, so: $\sum_{ij} (\Sigma_{ij}u)_{x_ix_j} = \sum_{ij} u_{x_ix_j}\Sigma_{ij}$. In particular, $\sigma$ is a real value, so $\sum_{ij} (\Sigma_{ij}u)_{x_ix_j} = \sum_{ij} u_{x_ix_j}\Sigma_{ij} = \sigma^2\Delta_x u$
- $\Delta_x u = \nabla_x \cdot \nabla_x u = \nabla_x \cdot \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) = \alpha u \Delta_x V(\left\|x\right\|^2) + \alpha \nabla_x u \cdot \nabla_x V(\left\|x\right\|^2) = \alpha u \Delta_x V(\left\|x\right\|^2) + \alpha \left(\alpha \nabla_x V(\left\|x\right\|^2)u\right) \cdot \nabla_x V(\left\|x\right\|^2) = \alpha u \Delta_x V(\left\|x\right\|^2) + \alpha^2u \left\|\nabla_x V(\left\|x\right\|^2)\right\|^2 $
- $u_t = - \sum_i (f_i u)_{x_i} + \frac12\sum_{ij} (\Sigma_{ij}u)_{x_ix_j} = - \left(-u\Delta_x V\left(\left\|x\right\|^2\right) - \alpha u \left\|\nabla_x V(\left\|x\right\|^2)\right\|^2\right) + \frac{\sigma^2}{2}\left(\alpha u \Delta_x V(\left\|x\right\|^2) + \alpha^2u \left\|\nabla_x V(\left\|x\right\|^2)\right\|^2\right) = u\left(\left(1+\frac{\alpha\sigma^2}{2}\right)\Delta_x V\left(\left\|x\right\|^2\right) + \alpha \left(1+\frac{\alpha\sigma^2}{2}\right)\left\|\nabla_x V(\left\|x\right\|^2)\right\|^2\right)$

We want $u_t=0$ and so $1+\frac{\alpha\sigma^2}{2}=0$.

The steady state is $\frac{1}{K}e^{-2V(\left\|x\right\|^2)/\sigma^2}$. $u$ is a probability density, so: $K=\int dx e^{-2V(\left\|x\right\|^2)/\sigma^2}$

In [8]:
# Test with 2D-Ring process

import numpy as np
from zhai2022.sde import Model
from zhai2022.sde.euler_maruyama import EulerMaruyama

R = np.array([[0,1],[-1,0]])

# Define drift, diffusion, and initial state functions
def drift(X, t):
    return - 4 * (np.linalg.norm(X, axis=-1, keepdims=True)**2-1) * X + np.einsum('ij, sj -> si', R, X)

def diffusion(X, t):
    return np.tile(np.eye(X.shape[1]), (X.shape[0], 1, 1))

def initial_state(n_samples):
    return np.ones((n_samples, 2)) * 0

# Create a Model instance
model = Model(drift, diffusion, initial_state, initial_time=0.0, n_dim=2)

dt_schedule = np.arange(1, 5001)**(-0.4) * 0.01
euler_maruyama = EulerMaruyama(model, dt_schedule)
X = euler_maruyama.get_trajectory(n_samples=1000)  # shape: (n_time_steps, n_samples, n_dim)

# make an animation
from matplotlib import pyplot as plt
from matplotlib import animation

T = np.sum(dt_schedule)
dt_mean = T / len(dt_schedule)

# make the animation
fig, ax = plt.subplots()
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal', 'box')
ax.set_title('2D Ring Process Simulation')

# scatter plot of samples
scat = ax.scatter(X[:, 0, 0], X[:, 0, 1], s=5, color='black', alpha=0.5, label='Samples')

# theoretical limit
X_theo = np.linspace(-3, 3, 300)
Y_theo = np.linspace(-3, 3, 300)
X_grid, Y_grid = np.meshgrid(X_theo, Y_theo)
Z_theo = np.exp(-2 * ((X_grid**2 + Y_grid**2 - 1)**2) / 1.0**2)
# color map for the theoretical limit
contour = ax.contourf(
    X_grid, Y_grid, Z_theo,
    levels=np.linspace(0, np.max(Z_theo), 10),
    cmap='viridis',
    alpha=0.6
)

ax.legend(
    loc='lower left',
    fontsize='small',
)

def update(frame):
    scat.set_offsets(X[frame, :, :])
    return scat,

ani = animation.FuncAnimation(fig, update, frames=X.shape[0], interval=int(dt_mean*1000), blit=True)
ani.save('2DRing_process.mp4', writer='ffmpeg', fps=int(1/dt_mean), dpi=400)
plt.close()